# Lab 8: Bahdanau Attention




Theory

Bahdanau Attention, also known as Additive Attention, was introduced by Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio (2014) to overcome the limitation of the traditional Sequence-to-Sequence (Seq2Seq) encoder-decoder model. In the original Seq2Seq architecture, the encoder compresses the entire input sentence into a single fixed-length context vector. This creates an information bottleneck, especially for long sentences, causing important information to be lost and reducing translation accuracy.

Bahdanau Attention solves this problem by allowing the decoder to focus on different parts of the input sequence while generating each output word. Instead of relying only on the final encoder hidden state, the decoder has access to all encoder hidden states and computes a weighted combination of them, called the context vector. The attention weights indicate the importance of each input word for predicting the current output word.

Working Principle

1.Encoder

A Bidirectional RNN (BiRNN) processes the input sentence. It generates a hidden state (annotation) for every input word. These hidden states collectively represent the entire source sentence.

2.Attention Mechanism

At each decoding step, the decoder compares its current hidden state with every encoder hidden state.

The attention (alignment) score is calculated as $e_{tj}=v^{T}\tanh(W_{1}h_{j}+W_{2}s_{t})$, where $h_j$ is the encoder hidden state of the $j^{th}$ input word, $s_t$ is the current decoder hidden state, and $W_1$, $W_2$, and $v$ are learnable parameters.

The scores are converted into attention weights using the Softmax function:

$\alpha_{tj}=\frac{\exp(e_{tj})}{\sum_{k}\exp(e_{tk})}$

The attention weights indicate how much importance should be given to each input word while predicting the current output word.

3.Context Vector

The context vector is computed as the weighted sum of all encoder hidden states:

$c_t=\sum_{j=1}^{T_x}\alpha_{tj}h_j$

Encoder hidden states with higher attention weights contribute more to the context vector.

4. Decoder
The decoder combines its current hidden state with the context vector.
It predicts the next output word.
The predicted word is fed back into the decoder for the next time step.
This process continues until the End-of-Sequence (EOS) token is generated.

**Requirements**

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data



- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input the the `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [5]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Now, let's bundle everything up as per the diagram above: 

In [6]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [7]:
PATH = r'data\eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes fort genereux', 'you re very generous']


15

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

### Encoder

In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.


### Decoder

In [9]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [10]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [11]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [12]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [13]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [15]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [16]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [17]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 5s (- 3m 52s) (5 2%) 1.9600
0m 9s (- 2m 58s) (10 5%) 1.2895
0m 12s (- 2m 40s) (15 7%) 1.1015
0m 16s (- 2m 28s) (20 10%) 0.9631
0m 20s (- 2m 20s) (25 12%) 0.8490
0m 23s (- 2m 14s) (30 15%) 0.7542
0m 27s (- 2m 8s) (35 17%) 0.6739
0m 30s (- 2m 3s) (40 20%) 0.6021
0m 34s (- 1m 58s) (45 22%) 0.5381
0m 37s (- 1m 53s) (50 25%) 0.4796
0m 41s (- 1m 49s) (55 27%) 0.4275
0m 44s (- 1m 44s) (60 30%) 0.3813
0m 48s (- 1m 40s) (65 32%) 0.3423
0m 51s (- 1m 36s) (70 35%) 0.3094
0m 55s (- 1m 32s) (75 37%) 0.2764
0m 59s (- 1m 28s) (80 40%) 0.2516
1m 2s (- 1m 24s) (85 42%) 0.2261
1m 6s (- 1m 21s) (90 45%) 0.2062
1m 10s (- 1m 17s) (95 47%) 0.1866
1m 13s (- 1m 13s) (100 50%) 0.1703
1m 17s (- 1m 10s) (105 52%) 0.1567
1m 21s (- 1m 6s) (110 55%) 0.1420
1m 24s (- 1m 2s) (115 57%) 0.1329
1m 28s (- 0m 59s) (120 60%) 0.1252
1m 32s (- 0m 55s) (125 62%) 0.1149
1m 35s (- 0m 51s) (130 65%) 0.1

In [18]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> elle est forte
= she is strong
< we re finally alone <EOS>

> je suis seulement fatigue
= i m just tired
< i m just tired <EOS>

> nous sommes tres excitees
= we re very excited
< we re very excited <EOS>

> nous descendons
= we re going down
< you re a prude <EOS>

> nous sommes tous impressionnes
= we re all impressed
< we re all impressed <EOS>

> il est malade
= he is ill
< she s still young <EOS>

> tu es tres intelligent
= you re very smart
< you re very intelligent <EOS>

> je suis contagieux
= i m contagious
< i m emotionally drained <EOS>

> tu es voyant
= you re psychic
< he s probably sleeping <EOS>

> nous sommes siderees
= we re stunned
< we re all done <EOS>



`Note`: Could train this with varying degree of success. 

Discussion

For this experiment, we built a Bahdanau Attention based RNN Encoder-Decoder model for neural machine translation. Unlike the simple Seq2Seq model, the decoder doesn’t use a single fixed-length context vector. Instead, it dynamically attends to all the encoder hidden states and computes a context vector at each decoding step using attention weights. This enables the model to pay attention to the most relevant words in the input sentence when predicting each target word, thereby alleviating the information bottleneck associated with long sentences. Figure 5 shows the loss during the training which decreased steadily and indicated that the model has learned the relationship between the source and target languages successfully. Using teacher forcing improved the training stability and convergence. Overall, the attention mechanism gave more accurate translations and showed better ability to handle.

Conclusion

This experiment showcased the use of the Bahdanau Attention (Additive Attention) mechanism in a Sequence-to-Sequence neural machine translation model. The encoder created hidden representations for each input word, while the decoder focused on these representations to produce each output word. The experiment revealed that the attention mechanism successfully addresses the fixed-length context vector problem of the traditional Seq2Seq model. This results in better translation quality and improved handling of long sentences. The lab offered a hands-on understanding of attention mechanisms, their role in neural machine translation, and their importance as the basis for modern Transformer-based architectures.